In [ ]:
import os
import ast
import pandas as pd
from tqdm import tqdm

In [ ]:
val_fold = 9
test_fold = 10

ptbxl_data = "/opt/gpudata/ecg/ptb-xl"
subset_root = "/opt/gpudata/ecg/temp" # path to make subset directories

In [ ]:
df = pd.read_csv(os.path.join(ptbxl_data, "ptbxl_database.csv"), index_col="ecg_id")
original_cols = df.columns

In [ ]:
df["strat_fold"].value_counts()

In [ ]:
def make_train_subset_from_fold(df: pd.DataFrame, train_folds: list[int]):
    # make subsets of training data
    subset = df[df["strat_fold"].isin(train_folds + [val_fold, test_fold])]
    train_size = subset["strat_fold"].value_counts().loc[train_folds].sum()

    train_size_k = train_size // 1024
    if train_size_k == 0:
        train_size_k = str(train_size)
    else:
        train_size_k = f"{train_size_k}k"
    subset_path = os.path.join(subset_root, f"ptb-xl-{train_size_k}")
    os.makedirs(subset_path, exist_ok=True)

    subset.to_csv(os.path.join(subset_path, "ptbxl_database.csv"))

    # also link source waveform data
    for p in [
        "records100",
        "records500",
    ]:
        os.symlink(
            src=os.path.join(ptbxl_data, p),
            dst=os.path.join(subset_path, p),
        )

In [ ]:
for train_folds in tqdm(
    [
        [1, 2, 3, 4],
        [1, 2],
        [1],
    ]
):
    make_train_subset_from_fold(df, train_folds)

### smaller PTB-XL subsets

difficult to use the same stratification algorithm developed by the PTB-XL authors to very small dataset sizes, so we use a coarser stratification strategy after exhausting the author provided splits:
* author provided "diagnostic" labels: NORM, MI, STTC, HYP, CD
* patient sex - same as author
* patient age (bins of 20 years) - same as author
to ensure stratification to the smallest subset (~2^8 samples), we bin samples with unique(-ish) labels together

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df["age_bin"] = pd.cut(df["age"], [0, 20, 40, 60, 80, 1000]).cat.codes # right edge of 1000 to account for age censoring >= 90

In [ ]:
# prepare coarse labels for stratification
agg_df = pd.read_csv(os.path.join(ptbxl_data, "scp_statements.csv"), index_col=0)
agg_df = agg_df[agg_df["diagnostic"] == 1]

df["scp_codes_raw"] = df["scp_codes"].copy()
df["scp_codes"] = df["scp_codes"].apply(lambda x: ast.literal_eval(x))

def aggregate_diagnostic(scp_codes: dict):
    tmp = []
    for key in scp_codes.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key, "diagnostic_class"])
    return list(set(tmp))

temp = df["scp_codes"].apply(aggregate_diagnostic)
temp = temp.apply(lambda x: pd.Series(1, index=x)).fillna(0).astype(int)

In [ ]:
# make subsets of first fold
mask = df["strat_fold"] == 1
splitting_cols = list(temp.columns) + ["sex", "age_bin"]
split_1_df = pd.concat([df, temp], axis=1).loc[mask]

In [ ]:
splitter = split_1_df[splitting_cols].apply(lambda row: "_".join([f"{x}" for x in row.values]), axis=1)

# set the samples with unique labels to an "other" group to enable splitting
# the starting subset (fold 1) is ~2k samples
# the final subset is ~256 samples - 1/8 the size
# so we need at least 8 samples to guarantee stratified sampling
threshold = 8
splitter_counts = splitter.value_counts()
to_group = splitter_counts[splitter_counts < threshold].index
print(f"Binning {splitter_counts[splitter_counts < threshold].sum()} (of {len(split_1_df)}) samples with unique labels together")
splitter.loc[splitter.isin(to_group)] = "other" # type: ignore

In [ ]:
def make_train_subset_small(_df: pd.DataFrame, _splitter: pd.Series, _name: str) -> tuple[pd.DataFrame, pd.Series]:
    _, subset_df, _, subset_splitter = train_test_split(_df, _splitter, test_size=0.5, random_state=42, shuffle=True, stratify=_splitter)
    subset_path = os.path.join(subset_root, f"ptb-xl-{_name}")
    os.makedirs(subset_path, exist_ok=True)

    # add back in val/test data to create a "full" dataset
    val_test_df = df[df["strat_fold"].isin({9, 10})]
    full_df = pd.concat([subset_df[original_cols], val_test_df[original_cols]])
    full_df.to_csv(os.path.join(subset_path, "ptbxl_database.csv"))

    # also link source waveform data
    for p in [
        "records100",
        "records500",
    ]:
        os.symlink(
            src=os.path.join(ptbxl_data, p),
            dst=os.path.join(subset_path, p),
        )
    return subset_df, subset_splitter

In [ ]:
_df, _splitter = split_1_df, splitter
# split 1 df is ~2k samples so first subset is ~1k
for _name in ["1k", "512", "256"]:
    _df, _splitter = make_train_subset_small(_df, _splitter, _name)